In [1]:
!pip install mysql-connector-python

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import mysql.connector

In [4]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Nandini1A@"
)

cursor = conn.cursor()

print("Connected Successfully!")

Connected Successfully!


In [5]:
cursor.execute("""
CREATE DATABASE IF NOT EXISTS email_deliverability_db
""")

print("Database created successfully!")

Database created successfully!


In [6]:
cursor.execute("USE email_deliverability_db")

print("Now using email_deliverability_db")

Now using email_deliverability_db


In [7]:
cursor.execute("SHOW DATABASES")

for db in cursor:
    print(db[0])

chatgpt
csv_files
customer_behaviour
email_deliverability_db
emp_new
employee_payroll_db
eticket_system
expense_intelligence
hospitaldb
hr
information_schema
it_service_desk
join_practice
music_store
mysql
netflix_data
performance_schema
sai
sakila
sys
tcl
trigger_demo
world
zepto_sql_project
zeta_practice


In [8]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_client(
    client_id INT PRIMARY KEY,
    client_tag_name VARCHAR(255),
    client_tag_num INT,
    has_valid_audience BOOLEAN
)
""")

print("dim_client created")

dim_client created


In [9]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_campaign(
    campaign_id INT PRIMARY KEY,
    campaign_test_id INT,
    campaign_test_part VARCHAR(50),
    campaign_test_type VARCHAR(100),
    campaign_subscribers_count INT,
    campaign_subscribers_blocked_count INT,
    campaign_moderation VARCHAR(50),
    campaign_landing_page BOOLEAN
)
""")

print("dim_campaign created")

dim_campaign created


In [10]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_message(
    message_id INT AUTO_INCREMENT PRIMARY KEY,
    message_content_size INT,
    message_images_count INT,
    message_embedded_images_count INT,
    message_embedded_images_size INT,
    message_embedded_files_count INT,
    message_embedded_files_size INT,
    message_links_count INT,
    message_links_blocked_count INT,
    personalization_tags_count INT
)
""")

print("dim_message created")

dim_message created


In [11]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS fact_deliverability(
    campaign_test_id INT PRIMARY KEY,
    client_id INT,
    campaign_id INT,
    message_id INT,
    sent_hour INT,
    sent_day_name VARCHAR(20),
    sent_month INT,
    sent_year INT,
    registrar_domain_number INT,
    domain_number INT,
    deliverability_status VARCHAR(20),

    FOREIGN KEY(client_id) REFERENCES dim_client(client_id),
    FOREIGN KEY(campaign_id) REFERENCES dim_campaign(campaign_id),
    FOREIGN KEY(message_id) REFERENCES dim_message(message_id)
)
""")

print("fact_deliverability created")

fact_deliverability created


In [12]:
conn.commit()
print("All tables created successfully!")

All tables created successfully!


In [13]:
import pandas as pd

df = pd.read_parquet(
    r"C:\Users\nukal\Downloads\AI_Email_Deliverability_Intelligence\content\drive\MyDrive\AI_Email_Deliverability_Intelligence\data\processed\sendguard_base_processed.parquet"
)

print(df.shape)
df.head(2)

(908226, 134)


,client_id,client_tag_name,client_tag_num,campaign_id,campaign_test_id,campaign_test_part,campaign_test_type,campaign_sent_time,message_content_size,message_images_count,...,open_rate,click_rate,sent_date,sent_hour,sent_day_of_week,sent_day_name,sent_month,sent_year,has_valid_audience,zero_audience_with_activity
0,171728,None,0,5995662,90351,part,sender_name,2024-04-01 02:00:07+00:00,40968,10,...,23.420074,3.345725,2024-04-01,2.0,0.0,Monday,4.0,2024.0,True,False
1,147025,None,0,5994558,0,0,0,2024-04-01 02:00:08+00:00,11330,5,...,21.201092,6.187443,2024-04-01,2.0,0.0,Monday,4.0,2024.0,True,False


In [14]:
client_df = (
    df[
        [
            "client_id",
            "client_tag_name",
            "client_tag_num",
            "has_valid_audience"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(client_df.shape)
client_df.head()

(4598, 4)


,client_id,client_tag_name,client_tag_num,has_valid_audience
0,171728,None,0,True
1,147025,None,0,True
2,463419,None,0,True
3,503371,None,0,True
4,481573,None,0,True


In [15]:
insert_query = """
INSERT IGNORE INTO dim_client
(client_id, client_tag_name, client_tag_num, has_valid_audience)
VALUES (%s, %s, %s, %s)
"""

cursor.executemany(
    insert_query,
    client_df.values.tolist()
)

conn.commit()

print("dim_client imported successfully!")

dim_client imported successfully!


In [16]:
cursor.execute("SELECT COUNT(*) FROM dim_client")

print("Rows in dim_client:", cursor.fetchone()[0])

Rows in dim_client: 4112


In [17]:
campaign_df = (
    df[
        [
            "campaign_id",
            "campaign_test_id",
            "campaign_test_part",
            "campaign_test_type",
            "campaign_subscribers_count",
            "campaign_subscribers_blocked_count",
            "campaign_moderation",
            "campaign_landing_page"
        ]
    ]
    .drop_duplicates(subset=["campaign_id"])
    .reset_index(drop=True)
)

print(campaign_df.shape)
campaign_df.head()

(908226, 8)


,campaign_id,campaign_test_id,campaign_test_part,campaign_test_type,campaign_subscribers_count,campaign_subscribers_blocked_count,campaign_moderation,campaign_landing_page
0,5995662,90351,part,sender_name,269,0,0,0
1,5994558,0,0,0,1099,0,0,0
2,5995663,90351,part,sender_name,269,0,0,0
3,5995628,0,0,0,24027,36,0,0
4,5995646,0,0,0,206,8,0,0


In [18]:
insert_query = """
INSERT IGNORE INTO dim_campaign
(
    campaign_id,
    campaign_test_id,
    campaign_test_part,
    campaign_test_type,
    campaign_subscribers_count,
    campaign_subscribers_blocked_count,
    campaign_moderation,
    campaign_landing_page
)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
"""

cursor.executemany(
    insert_query,
    campaign_df.values.tolist()
)

conn.commit()

print("dim_campaign imported successfully!")

dim_campaign imported successfully!


In [19]:
cursor.execute("SELECT COUNT(*) FROM dim_campaign")
print("Rows in dim_campaign:", cursor.fetchone()[0])

Rows in dim_campaign: 908226


In [20]:
message_df = (
    df[
        [
            "message_content_size",
            "message_images_count",
            "message_embedded_images_count",
            "message_embedded_images_size",
            "message_embedded_files_count",
            "message_embedded_files_size",
            "message_links_count",
            "message_links_blocked_count",
            "message_personalization_tags_count"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(message_df.shape)
message_df.head()

(542775, 9)


,message_content_size,message_images_count,message_embedded_images_count,message_embedded_images_size,message_embedded_files_count,message_embedded_files_size,message_links_count,message_links_blocked_count,message_personalization_tags_count
0,40968,10,0,0,0,0,11,0,0
1,11330,5,0,0,0,0,13,0,0
2,81182,21,0,0,0,0,9,0,1
3,52177,4,0,0,0,0,6,0,0
4,48404,8,0,0,0,0,12,0,0


In [21]:
insert_query = """
INSERT INTO dim_message(
    message_content_size,
    message_images_count,
    message_embedded_images_count,
    message_embedded_images_size,
    message_embedded_files_count,
    message_embedded_files_size,
    message_links_count,
    message_links_blocked_count,
    personalization_tags_count
)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
"""

cursor.executemany(
    insert_query,
    message_df.values.tolist()
)

conn.commit()

print("dim_message imported successfully!")

dim_message imported successfully!


In [22]:
cursor.execute("SELECT COUNT(*) FROM dim_message")
print("Rows in dim_message:", cursor.fetchone()[0])

Rows in dim_message: 542775


In [23]:
# Give every unique message a message_id
message_lookup = message_df.copy()
message_lookup["message_id"] = range(1, len(message_lookup) + 1)

# Merge with original dataset
fact_df = df.merge(
    message_lookup,
    on=[
        "message_content_size",
        "message_images_count",
        "message_embedded_images_count",
        "message_embedded_images_size",
        "message_embedded_files_count",
        "message_embedded_files_size",
        "message_links_count",
        "message_links_blocked_count",
        "message_personalization_tags_count"
    ],
    how="left"
)

print(fact_df.shape)
fact_df.head(2)

(908226, 135)


,client_id,client_tag_name,client_tag_num,campaign_id,campaign_test_id,campaign_test_part,campaign_test_type,campaign_sent_time,message_content_size,message_images_count,...,click_rate,sent_date,sent_hour,sent_day_of_week,sent_day_name,sent_month,sent_year,has_valid_audience,zero_audience_with_activity,message_id
0,171728,None,0,5995662,90351,part,sender_name,2024-04-01 02:00:07+00:00,40968,10,...,3.345725,2024-04-01,2.0,0.0,Monday,4.0,2024.0,True,False,1
1,147025,None,0,5994558,0,0,0,2024-04-01 02:00:08+00:00,11330,5,...,6.187443,2024-04-01,2.0,0.0,Monday,4.0,2024.0,True,False,2


In [24]:
def create_status(rate):
    if rate >= 99:
        return "Excellent"
    elif rate >= 95:
        return "Good"
    elif rate >= 90:
        return "Warning"
    else:
        return "Critical"

fact_df["deliverability_status"] = fact_df["delivery_rate"].apply(create_status)

In [25]:
fact_table = fact_df[
    [
        "campaign_test_id",
        "client_id",
        "campaign_id",
        "message_id",
        "sent_hour",
        "sent_day_name",
        "sent_month",
        "sent_year",
        "registrar_domain_number",
        "domain_number",
        "deliverability_status"
    ]
].copy()

print(fact_table.shape)
fact_table.head()

(908226, 11)


,campaign_test_id,client_id,campaign_id,message_id,sent_hour,sent_day_name,sent_month,sent_year,registrar_domain_number,domain_number,deliverability_status
0,90351,171728,5995662,1,2.0,Monday,4.0,2024.0,2,2,Excellent
1,0,147025,5994558,2,2.0,Monday,4.0,2024.0,1,1,Excellent
2,90351,171728,5995663,1,2.0,Monday,4.0,2024.0,2,2,Excellent
3,0,463419,5995628,3,2.0,Monday,4.0,2024.0,5,5,Excellent
4,0,503371,5995646,4,3.0,Monday,4.0,2024.0,3,3,Excellent


In [27]:
import numpy as np

fact_table = fact_table.replace({np.nan: None})

In [28]:
insert_query = """
INSERT IGNORE INTO fact_deliverability(
    campaign_test_id,
    client_id,
    campaign_id,
    message_id,
    sent_hour,
    sent_day_name,
    sent_month,
    sent_year,
    registrar_domain_number,
    domain_number,
    deliverability_status
)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
"""

cursor.executemany(
    insert_query,
    fact_table.values.tolist()
)

conn.commit()

print("fact_deliverability imported successfully!")

fact_deliverability imported successfully!


In [29]:
cursor.execute("SELECT COUNT(*) FROM fact_deliverability")
print("Rows:", cursor.fetchone()[0])

Rows: 6802


In [30]:
cursor.execute("DROP TABLE fact_deliverability")
conn.commit()

print("Old fact table deleted")

Old fact table deleted


In [31]:
cursor.execute("""
CREATE TABLE fact_deliverability(

    fact_id BIGINT AUTO_INCREMENT PRIMARY KEY,

    campaign_test_id INT,
    client_id INT,
    campaign_id INT,
    message_id INT,

    sent_hour INT,
    sent_day_name VARCHAR(20),
    sent_month INT,
    sent_year INT,

    registrar_domain_number INT,
    domain_number INT,

    deliverability_status VARCHAR(20),

    INDEX idx_campaign_test (campaign_test_id),
    INDEX idx_client (client_id),
    INDEX idx_campaign (campaign_id),

    FOREIGN KEY (client_id) REFERENCES dim_client(client_id),
    FOREIGN KEY (campaign_id) REFERENCES dim_campaign(campaign_id),
    FOREIGN KEY (message_id) REFERENCES dim_message(message_id)
)
""")

conn.commit()

print("New fact table created")

New fact table created


In [32]:
insert_query = """
INSERT INTO fact_deliverability(
    campaign_test_id,
    client_id,
    campaign_id,
    message_id,
    sent_hour,
    sent_day_name,
    sent_month,
    sent_year,
    registrar_domain_number,
    domain_number,
    deliverability_status
)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
"""

cursor.executemany(insert_query, fact_table.values.tolist())
conn.commit()

print("Imported successfully!")

Imported successfully!


In [33]:
cursor.execute("SELECT COUNT(*) FROM fact_deliverability")
print(cursor.fetchone()[0])

908226
